# In-room filtration scale-up

`src/filtration_scale_up_model.py` implements methods section 2.3. It
estimates global annual production of commercial portable air cleaners, DIY
Corsi-Rosenthal boxes and DIY units repurposed from coal-plant baghouse
filters, allocates that production across countries by manufacturing value
added, converts it to an equivalent clean air delivery rate (eCADR) and spreads
it over a weekly timeline. All three scenarios are run on every execution.

Every number lives in `data/filtration_scale_up/parameters_filtration_scale_up.csv`.
Rows with a distribution are uncertain and sampled; the rest are fixed
settings. Nothing is hard-coded in the model.

Run the scripts from the repository root, in this order:

```bash
python src/essential_workers.py                          # workforce requirements
python src/processing/linear_models.py                   # fits, updates the parameters file
python src/filtration_scale_up_model.py                  # results/filtration_scale_up/
python scripts/visualization/plot_filtration_coverage.py  # the filtration figures
```

Both model scripts stop rather than run on incomplete inputs.
`linear_models.py` refuses while `data/filtration_scale_up/coal_plant_airflow.csv`
is empty, and the model refuses while any uncertain parameter still has
`low == high == 0`, naming each one. Manufacturing value added is downloaded
from the World Bank on the first run and cached to
`data/filtration_scale_up/mva_world_bank.csv`; check that the printed country
count is roughly 190-200, since a much lower number means the download failed
and a stale cache was used.

This notebook walks through the model one stage at a time. It needs
`results/essential_workers/essential_workers_by_country.csv`, which
`src/essential_workers.py` produces.


## 0. Setup

Imports work from the repository root or from `scripts/`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO / 'scripts' / 'visualization'))

import filtration_scale_up_model as sm
import plot_filtration_coverage as pfc
import processing.linear_models as lm
from processing.paths import (
    CR_BOXES_PRIORITIZED_RESULTS,
    FILTRATION_SCALE_UP_PARAMETERS,
    FILTRATION_SCALE_UP_VISUALIZATIONS,
    PACS_PRIORITIZED_RESULTS,
    read_parameters,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

settings, params = read_parameters(FILTRATION_SCALE_UP_PARAMETERS)
print(f'{len(params)} uncertain parameters, {len(settings)} fixed settings')
print(f"{int(settings['n_draws'])} draws over {int(settings['weeks'])} weeks")


## 1. Inputs

The parameters file gives a low and a high for every uncertain quantity, with
the distribution to sample between them, and a single value for every fixed
setting. `read_parameters` refuses to return while any uncertain row is still
unset, so a missing number fails the run rather than quietly defaulting to zero.


In [ ]:
display(params.head(10))
display(pd.Series(settings, name='value').to_frame())


## 2. Fitted inputs

Two settings come from regressions rather than the literature: the coal
baghouse airflow per megawatt (methods equation 7) and the exponent that
allocates production by manufacturing value added (methods equation 3).
`linear_models.py` fits both and writes them into the parameters file with the R2
and sample size they came from, so there is nothing to copy by hand.

In [ ]:
coal = lm.fit_coal_airflow()
pooled = lm.fit_allocator()['pooled']
print(f"baghouse airflow: {coal['slope']:,.0f} L/s per MW + {coal['intercept']:,.0f} L/s "
      f"per plant (R2 = {coal['r_squared']:.2f}, n = {coal['n']})")
print(f'MVA exponent b:   {pooled.params[1]:.3f} +/- {pooled.bse[1]:.3f} '
      f'(R2 = {pooled.rsquared:.3f}, n = {int(pooled.nobs)})')
print(f'The parameters file bounds b from {params.loc["mva_exponent_b", "low"]} '
      f'to {params.loc["mva_exponent_b", "high"]}')

## 3. Countries and the allocation of production

The country table joins the workforce requirements from the essential worker
pipeline to World Bank manufacturing value added and to installed coal
capacity. Global production is then split between countries in proportion to
manufacturing value added raised to the fitted exponent. Countries whose
implied output falls below the minimum viable factory size are zeroed and their
share is spread over the rest.

In [ ]:
np.random.seed(int(settings['random_seed']))
n = int(settings['n_draws'])
samples = sm.sample_all(params, n)
df = sm.load_country_data(bool(settings.get('adjust_MVA_by_cost')))

total_filters = sm.filter_production(
    samples, samples['total_air_filter_revenue_usd'], sm.MERV_BANDS
)
shares = np.median(sm.production_shares(
    df,
    samples['mva_exponent_b'],
    settings['min_country_filter_production'],
    total_filters,
), axis=0)
top = pd.DataFrame({'country': df['Country Name'], 'share': shares})
display(top.nlargest(10, 'share').style.format({'share': '{:.1%}'}))

## 4. Supply channels

Six channels carry eCADR. Three are manufactured over the timeline and grow
with the scenario multiplier; three are one-off transfers of units already in
service, released over a few weeks.

| Channel | Methods | What it is |
| --- | --- | --- |
| `pac` | eq. 1-2 | New commercial portable air cleaners |
| `cr_box` | eq. 4-6 | New Corsi-Rosenthal boxes, limited by filters or fans |
| `baghouse` | eq. 7-8 | New coal baghouse replacement bags |
| `repurposed_pac` | eq. 9 | Portable air cleaners pulled from non-essential workplaces |
| `repurposed_cr_box` | eq. 10 | Non-residential panel filters pulled from non-essential workplaces |
| `repurposed_baghouse` | eq. 8 | Bags recovered from coal plants left idle |

In [ ]:
streams = sm.build_streams(df, samples, settings, n, scenario=2, shares=shares)
weeks = int(settings['weeks'])
at_end = pd.Series(
    {name: np.median(stream[-1].sum(axis=1)) for name, stream in streams.items()},
    name=f'eCADR at week {weeks} (L/s)',
)
display(at_end.to_frame().style.format('{:,.3e}'))

## 5. Scenarios

Scenario 1 is the increase in utilisation of existing factories alone.
Scenario 3 is COVID-analogue growth in production. Scenario 2 is scenario 3
capped by the extra filters the available meltblown polypropylene supports, so
it lies between the two. Coal baghouse bags are woven synthetics or glass
rather than meltblown polypropylene, so that cap does not apply to them. This
is an assumption, as there could be competition in reality.

All scenarios share the same parameter draws, so they can be compared draw for
draw.

In [ ]:
requirement = df['Indoor Vital CADR Requirement (L/s)'].to_numpy(float)
comparison = {}
for scenario, description in sm.SCENARIOS.items():
    total = sum(sm.build_streams(df, samples, settings, n, scenario, shares).values())
    result = sm.coverage(total, requirement, df, settings['uncertainty_interval'])
    globally = result[result.region == 'Global'].set_index('week')
    comparison[f'{scenario}: {description}'] = {
        f'week {week}': globally.loc[week, 'coverage_median'] for week in sm.REPORT_WEEKS
    }
display(pd.DataFrame(comparison).T.style.format('{:.1%}'))

## 6. Run the model and draw the figures

`main()` runs all three scenarios twice and writes
`results/filtration_scale_up/pacs_prioritized/` (default) and
`results/filtration_scale_up/cr_boxes_prioritized/`: weekly eCADR by country
and by channel, coverage of both workforces by region and week, and the
requirement each region is measured against. The figure script then reads
those files and writes `results/filtration_scale_up/visualizations/`.


In [ ]:
sm.main()

In [ ]:
pfc.main(FILTRATION_SCALE_UP_VISUALIZATIONS, scenario=2)


In [ ]:
for name in [
    'scenario_coverage_covid_measles.png',
    'global_stacked_cadr.png',
    'supply_and_coverage_week26.png',
]:
    display(Image(filename=str(FILTRATION_SCALE_UP_VISUALIZATIONS / name), width=760))


## 7. Re-run the test suite

```bash
pytest tests/test_filtration_scale_up_model.py -q
```


## 8. Key results

Numbers for the in-room filtration scale-up results section of the manuscript.
Reads `results/filtration_scale_up/pacs_prioritized/` (default) and
`cr_boxes_prioritized/` for the panel-filter sensitivity. Re-run
`python src/filtration_scale_up_model.py` if those CSVs are missing or stale.


In [ ]:
WEEK = 26
REPORT_SCENARIO = 2  # meltblown-capped growth (Figure 4 / 5)

requirements = pfc.load_requirements()
vital_requirement = requirements.loc['Global', 'indoor_vital_ecadr_l_per_s']
vital_requirement_covid = requirements.loc['Global', 'indoor_vital_ecadr_covid_l_per_s']
ui_pct = int(settings['uncertainty_interval'])

ui_label = f"{ui_pct}% UI"


def load_coverage_table(label, scenario, pathogen='measles', results_dir=PACS_PRIORITIZED_RESULTS):
    """Read the coverage CSV for one workforce, scenario and pathogen ('measles' or 'covid')."""
    suffix = '_covid' if pathogen == 'covid' else ''
    return pd.read_csv(results_dir / f"coverage_{label}{suffix}_scenario{scenario}.csv")


def load_global_coverage(label, scenario, week, pathogen='measles', results_dir=PACS_PRIORITIZED_RESULTS):
    """Read global median coverage and uncertainty interval for one scenario."""
    table = load_coverage_table(label, scenario, pathogen, results_dir)
    row = table.query("region == 'Global' and week == @week").iloc[0]
    return {
        'median': 100.0 * row['coverage_median'],
        'lower': 100.0 * row['coverage_lower'],
        'upper': 100.0 * row['coverage_upper'],
    }


def fmt_coverage(stats):
    """Format median and uncertainty interval for manuscript text."""
    return (
        f"{stats['median']:.1f}% ({ui_label}: "
        f"{stats['lower']:.1f}–{stats['upper']:.1f}%)"
    )


def channel_shares_of_requirement(channels, week, requirement):
    """Each channel as a percent of the indoor vital worker requirement."""
    row = channels.loc[week]
    return 100.0 * row / requirement


def channel_shares_of_total(channels, week):
    """Each channel as a percent of total eCADR."""
    row = channels.loc[week]
    return 100.0 * row / row.sum()


channel_labels = {
    'cr_box': 'CR box manufacturing',
    'pac': 'PAC manufacturing',
    'repurposed_pac': 'PAC repurposing',
    'repurposed_cr_box': 'CR box repurposing',
    'baghouse': 'Coal baghouse manufacturing',
    'repurposed_baghouse': 'Coal baghouse repurposing',
}

pathogen_text = [('measles', 'Measles-like'), ('covid', 'COVID-like')]

scenario_text = [
    (1, 'Factory utilisation only (75% to 100%)'),
    (3, 'COVID-analogue growth (2–6x over two months)'),
    (2, 'COVID-analogue growth capped by meltblown supply'),
]

print('=' * 72)
print('IN-ROOM FILTRATION SCALE-UP')
print('=' * 72)

for pathogen, pathogen_label in pathogen_text:
    print(
        f'\n{pathogen_label} global coverage after 6 months (week {WEEK}), '
        'as a share of indoor CADR requirement:'
    )
    for scenario, label in scenario_text:
        vital = load_global_coverage('vital', scenario, WEEK, pathogen)
        essential = load_global_coverage('essential', scenario, WEEK, pathogen)
        print(f"\n{label}")
        print(f"  Indoor vital workers: {fmt_coverage(vital)}")
        print(f"  Indoor essential workers: {fmt_coverage(essential)}")

print(
    f"\nMeasles-like global indoor vital coverage at week {WEEK} by mask "
    "efficiency (meltblown-capped scenario):"
)
vital = load_global_coverage('vital', REPORT_SCENARIO, WEEK)
for mask, ratio in pfc.mask_requirement_ratios().items():
    scaled = {key: value / ratio for key, value in vital.items()}
    print(f"  {mask:.0%} efficiency masks: {fmt_coverage(scaled)}")

channels = pd.read_csv(
    PACS_PRIORITIZED_RESULTS / f'ecadr_by_channel_scenario{REPORT_SCENARIO}.csv',
    index_col='week',
)
shares_req = channel_shares_of_requirement(channels, WEEK, vital_requirement)
shares_req_covid = channel_shares_of_requirement(channels, WEEK, vital_requirement_covid)
shares_total = channel_shares_of_total(channels, WEEK)

print(
    f"\nAt week {WEEK} under meltblown-capped growth, channel shares of the "
    f"indoor vital requirement (measles-like, COVID-like):"
)
for key, label in channel_labels.items():
    print(f"  {label}: {shares_req[key]:.1f}%, {shares_req_covid[key]:.1f}%")
print(f"  Total: {shares_req.sum():.1f}%, {shares_req_covid.sum():.1f}%")

channels_cr = pd.read_csv(
    CR_BOXES_PRIORITIZED_RESULTS / f'ecadr_by_channel_scenario{REPORT_SCENARIO}.csv',
    index_col='week',
)
shares_total_default = channel_shares_of_total(channels, WEEK)
shares_total_cr = channel_shares_of_total(channels_cr, WEEK)
total_default = channels.loc[WEEK].sum()
total_cr = channels_cr.loc[WEEK].sum()

print('\nPanel filters diverted to CR boxes vs PACs prioritized (default):')
print(
    f"  PAC manufacturing: {shares_total_cr['pac']:.1f}% of eCADR "
    f"(CR boxes prioritized) vs {shares_total_default['pac']:.1f}% (default)"
)
print(
    f"  Total eCADR change: {100.0 * (total_default - total_cr) / total_cr:+.1f}%"
)
for pathogen, pathogen_label in pathogen_text:
    coverage_default = load_global_coverage('vital', REPORT_SCENARIO, WEEK, pathogen)
    coverage_cr = load_global_coverage(
        'vital', REPORT_SCENARIO, WEEK, pathogen, results_dir=CR_BOXES_PRIORITIZED_RESULTS
    )
    print(
        f"  {pathogen_label} global vital coverage: {fmt_coverage(coverage_cr)} "
        f"(CR boxes prioritized) vs {fmt_coverage(coverage_default)} (default)"
    )

# Regional production shares
if 'shares' not in globals() or 'df' not in globals():
    np.random.seed(int(settings['random_seed']))
    n = int(settings['n_draws'])
    samples = sm.sample_all(params, n)
    df = sm.load_country_data(bool(settings.get('adjust_MVA_by_cost')))
    total_filters = sm.filter_production(
        samples, samples['total_air_filter_revenue_usd'], sm.MERV_BANDS
    )
    shares = np.median(sm.production_shares(
        df,
        samples['mva_exponent_b'],
        settings['min_country_filter_production'],
        total_filters,
    ), axis=0)

production_by_region = (
    pd.DataFrame({'region': df['region'], 'share': shares})
    .groupby('region', as_index=False)['share']
    .sum()
    .sort_values('share', ascending=False)
)
production_by_region['share %'] = 100.0 * production_by_region['share']

print(f"\nRegional shares of global filter production:")
for _, row in production_by_region.iterrows():
    print(f"  {row['region']}: {row['share %']:.1f}%")

coverage_maps = {}
for pathogen, pathogen_label in pathogen_text:
    coverage_vital = load_coverage_table('vital', REPORT_SCENARIO, pathogen)
    coverage_map = (
        coverage_vital.loc[coverage_vital['week'] == WEEK]
        .query("region != 'Global'")
        .assign(coverage_pct=lambda d: 100.0 * d['coverage_median'])
        .sort_values('coverage_pct', ascending=False)
    )
    coverage_maps[pathogen] = coverage_map

    print(
        f"\n{pathogen_label} indoor vital worker coverage at week {WEEK} "
        f"(meltblown-capped scenario), by UN region:"
    )
    for _, row in coverage_map.iterrows():
        stats = {
            'median': row['coverage_pct'],
            'lower': 100.0 * row['coverage_lower'],
            'upper': 100.0 * row['coverage_upper'],
        }
        print(f"  {row['region']}: {fmt_coverage(stats)}")

print('\n--- Tables for copy/paste ---')
display(production_by_region.reset_index(drop=True))
for pathogen, coverage_map in coverage_maps.items():
    display(coverage_map[['region', 'coverage_pct']].rename(
        columns={'coverage_pct': f'{pathogen} coverage %'}
    ).reset_index(drop=True))
display(pd.DataFrame({
    'channel': [channel_labels[k] for k in channel_labels],
    '% of vital requirement (measles)': [shares_req[k] for k in channel_labels],
    '% of vital requirement (COVID)': [shares_req_covid[k] for k in channel_labels],
    '% of total eCADR (default)': [shares_total_default[k] for k in channel_labels],
}))
